# 06 — MLP / Feed-Forward Networks

This notebook builds from a small multi-layer perceptron to a wider feed-forward
network, then connects the implementation to tensor shapes and linear algebra.

## Experiments

1. `2 → 2 → 1` MLP with fixed weights
2. XOR with an MLP
3. ReLU² FFN with `768 → 3072 → 768`
4. AND and OR with MLPs
5. Deeper `5 → 8 → 16 → 1` MLP
6. Linear algebra reference: tensors, transpose, and matrix multiplication

## 0 — Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch version:", torch.__version__)

## 1 — A `2 → 2 → 1` MLP

```text
2 inputs → 2 hidden units → ReLU → 1 output
```

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.relu = nn.ReLU()
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

    def setup(self):
        with torch.no_grad():
            self.hidden.weight.copy_(torch.tensor([[1.2, 0.5], [-0.5, 1.0]]))
            self.hidden.bias.copy_(torch.tensor([0.0, 0.3]))
            self.output.weight.copy_(torch.tensor([[1.0, 0.8]]))
            self.output.bias.copy_(torch.tensor([0.0]))

model = SimpleMLP()
model.setup()

x = torch.tensor([1.0, 0.5])
print("Input:", x)
print("Output:", model(x))
print("Expected output: 1.69")

## 2 — XOR with an MLP

In [ ]:
class LogicMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.relu = nn.ReLU()
        self.output = nn.Linear(2, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

xor_model = LogicMLP()

with torch.no_grad():
    xor_model.hidden.weight.copy_(torch.tensor([[1.0, 1.0], [1.0, 1.0]]))
    xor_model.hidden.bias.copy_(torch.tensor([-0.5, -1.5]))
    xor_model.output.weight.copy_(torch.tensor([[2.0, -6.0]]))
    xor_model.output.bias.copy_(torch.tensor([0.0]))

print("A | B | XOR")
print("--+---+----")
for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    y = xor_model(torch.tensor([float(a), float(b)])).item()
    print(f"{a} | {b} | {y:.0f}")

## 3 — ReLU² Feed-Forward Network

```text
768 → 3072 → ReLU² → 768
```

The exercise uses `bias=False` for both linear projections.

In [ ]:
class ReluSquaredFFN(nn.Module):
    def __init__(self, input_dimension=768, upsample_ratio=4):
        super().__init__()
        hidden_dimension = input_dimension * upsample_ratio

        self.upsample_weights = nn.Linear(
            input_dimension, hidden_dimension, bias=False
        )
        self.downsample_weights = nn.Linear(
            hidden_dimension, input_dimension, bias=False
        )

    def forward(self, x):
        x = self.upsample_weights(x)
        x = F.relu(x)
        x = x.square()
        x = self.downsample_weights(x)
        return x

ffn = ReluSquaredFFN(input_dimension=768, upsample_ratio=4)
print(ffn)

x = torch.randn(768)
y = ffn(x)

print("Input shape: ", tuple(x.shape))
print("Output shape:", tuple(y.shape))
print("Parameter count:", sum(p.numel() for p in ffn.parameters()))

## 4 — AND and OR with MLPs

In [ ]:
def configure_and_model():
    model = LogicMLP()
    with torch.no_grad():
        model.hidden.weight.copy_(torch.tensor([[1.0, 1.0], [0.0, 0.0]]))
        model.hidden.bias.copy_(torch.tensor([-1.5, 0.0]))
        model.output.weight.copy_(torch.tensor([[2.0, 0.0]]))
        model.output.bias.copy_(torch.tensor([0.0]))
    return model

def configure_or_model():
    model = LogicMLP()
    with torch.no_grad():
        model.hidden.weight.copy_(torch.tensor([[1.0, 1.0], [1.0, 1.0]]))
        model.hidden.bias.copy_(torch.tensor([-0.5, -1.5]))
        model.output.weight.copy_(torch.tensor([[2.0, -4.0]]))
        model.output.bias.copy_(torch.tensor([0.0]))
    return model

and_model = configure_and_model()
or_model = configure_or_model()

print("A | B | AND | OR")
print("--+---+------+---")
for a, b in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    x = torch.tensor([float(a), float(b)])
    and_y = and_model(x).item()
    or_y = or_model(x).item()
    print(f"{a} | {b} |  {and_y:.0f}   | {or_y:.0f}")

## 5 — A deeper `5 → 8 → 16 → 1` MLP

```text
5 → 8 → ReLU → 16 → ReLU → 1
```

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, input_dim=5, hidden1_dim=8, hidden2_dim=16, output_dim=1):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden1_dim, bias=True)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(hidden1_dim, hidden2_dim, bias=True)
        self.relu2 = nn.ReLU()
        self.output_layer = nn.Linear(hidden2_dim, output_dim, bias=True)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.output_layer(x)
        return x

model = DeepMLP()
x = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5], dtype=torch.float32)

with torch.no_grad():
    y = model(x)

print(model)
print("Input shape:", tuple(x.shape))
print("Output shape:", tuple(y.shape))
print("Output:", y.item())
print("Parameter count:", sum(p.numel() for p in model.parameters()))

### Parameter count

For `5 → 8 → 16 → 1`:

- `5 → 8`: `5×8 + 8 = 48`
- `8 → 16`: `8×16 + 16 = 144`
- `16 → 1`: `16×1 + 1 = 17`

Total: **209 trainable parameters**.

## 6 — Linear Algebra Reference

In [ ]:
tensor_1d = torch.tensor([1, 2, 3, 4, 5])

tensor_2d = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

print("1D tensor:")
print(tensor_1d)

print("\n2D tensor:")
print(tensor_2d)
print("Shape:", tensor_2d.shape)

transposed_tensor_2d = tensor_2d.T

print("\nTransposed 2D tensor:")
print(transposed_tensor_2d)
print("Shape:", transposed_tensor_2d.shape)

In [ ]:
matrix_a = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

matrix_b = torch.tensor([
    [7, 8],
    [9, 10],
    [11, 12]
])

matmul_result = matrix_a @ matrix_b

print("A shape:", matrix_a.shape)
print("B shape:", matrix_b.shape)
print("A @ B shape:", matmul_result.shape)
print("\nA @ B:")
print(matmul_result)

In [ ]:
vector = torch.tensor([1, 2])

matrix_c = torch.tensor([
    [1, 2, 3],
    [4, 5, 6]
])

result_vec_mat = vector @ matrix_c

print("Vector shape:", vector.shape)
print("Matrix shape:", matrix_c.shape)
print("Result shape:", result_vec_mat.shape)
print("Result:", result_vec_mat)

# Observations

- A perceptron with multiple inputs is naturally expressed as a vector dot product.
- An entire fully connected layer can be represented with a weight matrix.
- `nn.Linear` performs the core learned linear transformation in an MLP.
- Nonlinear activation functions are required between linear layers if the network
  is to represent nonlinear mappings.
- The XOR, AND, and OR exercises show how weights and biases control network behavior.
- Tensor dimensions provide a concise description of information flow through a model.
- The ReLU² FFN expands `768 → 3072`, applies a nonlinear transformation, and projects
  the representation back to `768`.
- The `5 → 8 → 16 → 1` example contains 209 trainable parameters.
- Matrix multiplication is central to fully connected layers and larger neural networks.

# Connection to the LLM Roadmap

```text
Perceptron
    ↓
Activation function
    ↓
MLP / FFN
    ↓
Later transformer components
```

The same basic pattern — linear projection, nonlinear transformation, and another
projection — will reappear inside larger language-model architectures.